# BÁO CÁO THỰC HÀNH LAB 04
- **Họ và tên:** Nguyễn Văn Phát
- **MSSV:** 24521314
- **Môn học:** Học máy Thống kê (DS102)
- **Bộ dữ liệu:** openfda_drug_events (Dữ liệu tự thu thập từ Lab 01, chia tập train.csv và test.csv)

## Assignment 3:

- Using Numpy to implement the soft-margin SVM model. 
- Train this model using SGD method.
- Evaluate this model using Precision, Recall, and F1 metrics.
*(Thực hiện trên tập train.csv và test.csv từ bộ dữ liệu Lab 01)*

### 1. Chuẩn bị dữ liệu từ train.csv và test.csv

In [1]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import precision_score, recall_score, f1_score, accuracy_score
from sklearn.linear_model import SGDClassifier

# Doc du lieu tap train va test da chia san
train = pd.read_csv('train.csv')
test = pd.read_csv('test.csv')

features = ['Age', 'Weight_kg', 'Sex', 'Num_Drugs']
target = 'Hospitalization'

X_train = train[features].values
# Chuyen nhan: 1 -> 1, 0 -> -1 theo cong thuc SVM
y_train = np.where(train[target].values == 1, 1.0, -1.0).astype(np.float32)

X_test = test[features].values
y_test = np.where(test[target].values == 1, 1.0, -1.0).astype(np.float32)

# Chuan hoa dac trung
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

print("Kích thước tập train:", X_train.shape, y_train.shape)
print("Kích thước tập test: ", X_test.shape, y_test.shape)

Kích thước tập train: (5192, 4) (5192,)
Kích thước tập test:  (1298, 4) (1298,)


### 2. Cài đặt mô hình Soft-margin SVM bằng SGD (NumPy)

In [2]:
class SoftMarginSVM:
    def __init__(self, lambda_param=0.01, lr=0.005, epochs=50):
        self.lambda_param = lambda_param
        self.lr = lr
        self.epochs = epochs
        self.w = None
        self.b = 0.0

    def fit(self, X, y):
        n_samples, n_features = X.shape
        self.w = np.zeros(n_features, dtype=np.float32)
        self.b = 0.0

        for epoch in range(self.epochs):
            # Giam nhe learning rate theo epoch
            lr = self.lr / (1 + 0.001 * epoch)
            for i in range(n_samples):
                xi = X[i]
                yi = y[i]
                margin = yi * (np.dot(xi, self.w) + self.b)
                
                # Cap nhat subgradient theo ham Hinge loss
                if margin < 1:
                    self.w = (1 - lr * self.lambda_param) * self.w + lr * yi * xi
                    self.b = self.b + lr * yi
                else:
                    self.w = (1 - lr * self.lambda_param) * self.w

    def predict(self, X):
        return np.sign(np.dot(X, self.w) + self.b)

### 3. Huấn luyện và đánh giá trên tập test

In [3]:
# Huan luyen mo hinh
model = SoftMarginSVM(lambda_param=0.01, lr=0.005, epochs=50)
model.fit(X_train, y_train)

# Du doan tren tap test
y_pred_sgd = model.predict(X_test)

# Chuyen ve nhan 0 va 1 de tinh chi so
y_test_bin = (y_test == 1).astype(int)
y_pred_bin_sgd = (y_pred_sgd == 1).astype(int)

p_sgd = precision_score(y_test_bin, y_pred_bin_sgd)
r_sgd = recall_score(y_test_bin, y_pred_bin_sgd)
f1_sgd = f1_score(y_test_bin, y_pred_bin_sgd)
acc_sgd = accuracy_score(y_test_bin, y_pred_bin_sgd)

print("--- Kết quả đánh giá Soft-margin SVM (NumPy SGD) ---")
print(f"Accuracy:  {acc_sgd:.4f}")
print(f"Precision: {p_sgd:.4f}")
print(f"Recall:    {r_sgd:.4f}")
print(f"F1-score:  {f1_sgd:.4f}")

--- Kết quả đánh giá Soft-margin SVM (NumPy SGD) ---
Accuracy:  0.5871
Precision: 0.5922
Recall:    0.5593
F1-score:  0.5753


## Assigment 4:
- Implement SVM method using a machine learning library (such as sklearn or sktorch).
- Train this model on the dataset.
- Evaluate this model using Precision, Recall, and F1 metrics.
- Compare the results of SVM using library to those of implemented SVM.
*(Thực hiện trên tập train.csv và test.csv từ bộ dữ liệu Lab 01)*

### 1. Cài đặt mô hình SVM bằng thư viện Scikit-Learn

In [4]:
# Su dung SGDClassifier voi loss='hinge' va penalty='l2' (tuong duong Soft-margin SVM)
clf = SGDClassifier(loss='hinge', penalty='l2', alpha=0.01, max_iter=50, random_state=42)
clf.fit(X_train, y_train)

# Du doan tren tap test
y_pred_sk = clf.predict(X_test)
y_pred_bin_sk = (y_pred_sk == 1).astype(int)

p_sk = precision_score(y_test_bin, y_pred_bin_sk)
r_sk = recall_score(y_test_bin, y_pred_bin_sk)
f1_sk = f1_score(y_test_bin, y_pred_bin_sk)
acc_sk = accuracy_score(y_test_bin, y_pred_bin_sk)

print("--- Kết quả đánh giá Scikit-Learn SVM (SGDClassifier) ---")
print(f"Accuracy:  {acc_sk:.4f}")
print(f"Precision: {p_sk:.4f}")
print(f"Recall:    {r_sk:.4f}")
print(f"F1-score:  {f1_sk:.4f}")

--- Kết quả đánh giá Scikit-Learn SVM (SGDClassifier) ---
Accuracy:  0.6032
Precision: 0.6255
Recall:    0.5146
F1-score:  0.5647


### 2. So sánh kết quả giữa hai phương pháp

In [5]:
df_compare = pd.DataFrame({
    'Mô hình': ['NumPy SGD SVM', 'Scikit-Learn SVM'],
    'Accuracy': [round(acc_sgd, 4), round(acc_sk, 4)],
    'Precision': [round(p_sgd, 4), round(p_sk, 4)],
    'Recall': [round(r_sgd, 4), round(r_sk, 4)],
    'F1-score': [round(f1_sgd, 4), round(f1_sk, 4)]
})

print("--- Bảng so sánh kết quả giữa hai phương pháp ---")
print(df_compare.to_string(index=False))

--- Bảng so sánh kết quả giữa hai phương pháp ---
         Mô hình  Accuracy  Precision  Recall  F1-score
   NumPy SGD SVM    0.5871     0.5922  0.5593    0.5753
Scikit-Learn SVM    0.6032     0.6255  0.5146    0.5647


### Nhận xét so sánh:
- Cả hai mô hình đều đạt hiệu năng xấp xỉ tương đương nhau trên tập kiểm tra (Accuracy ~59-60%, F1-score ~56-57%).
- Mô hình Soft-margin SVM tự cài đặt bằng NumPy với thuật toán SGD theo hàm Hinge loss có kết quả Recall cao hơn một chút so với Scikit-Learn (~56% so với ~51%), trong khi Scikit-Learn có Precision nhỉnh hơn (~63% so với ~59%).
- Việc áp dụng giải thuật SGD giúp mô hình hội tụ nhanh trên tập dữ liệu bảng và mô phỏng chính xác cơ chế tối ưu hóa của hàm mất mát Hinge loss.